<a href="https://colab.research.google.com/github/Wuanzz/FER-Video-Emotion-Recognition/blob/main/demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cell 1: Cài đặt các thư viện cần thiết

In [93]:
!pip uninstall -y opencv-python opencv-python-headless opencv-contrib-python opencv-contrib-python-headless -q
!pip install -q opencv-python-headless==4.10.0.84 gradio pandas

print("========================================")
print("✅ Cài đặt thư viện hoàn tất!")
print("========================================")

✅ Cài đặt thư viện hoàn tất!


# Cell 2: Kết nối Google Drive

In [94]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Đã kết nối Google Drive!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Đã kết nối Google Drive!


# Cell 3: Import thư viện & Kiểm tra Môi trường

In [95]:
import os
import cv2
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torchvision import models, transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ PyTorch Device: {device}")

✅ PyTorch Device: cuda


# Cell 4: Cấu hình Dự án & Đường dẫn File Checkpoint

In [96]:
PROJECT_DIR = "/content/drive/MyDrive/Công nghệ phần mềm nâng cao/Project_FER_Video"
MODEL_PATH = os.path.join(PROJECT_DIR, "checkpoints", "best_model.pth")

CLASS_NAMES = ['neutral', 'calm', 'happy', 'sad', 'angry', 'fearful', 'disgust', 'surprised']
IMAGE_SIZE = (224, 224)
SEQUENCE_LENGTH = 16

# Cell 5: Khởi tạo OpenCV Haar Cascade Face Detector

In [97]:
cascade_path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
face_cascade = cv2.CascadeClassifier(cascade_path)

if face_cascade.empty():
    raise RuntimeError("⚠️ Không thể tải file XML Haar Cascade!")
print("✅ Haar Cascade Face Detector sẵn sàng!")

✅ Haar Cascade Face Detector sẵn sàng!


# Cell 6: Định nghĩa Kiến trúc Mô hình FER

In [98]:
class SpatialTemporalFERModel(nn.Module):
    def __init__(self, backbone_name="resnet18", rnn_type="LSTM", hidden_dim=256,
                 num_rnn_layers=2, dropout=0.5, num_classes=8):
        super().__init__()

        if backbone_name == "resnet18":
            resnet = models.resnet18(weights=None)
            feature_dim = resnet.fc.in_features
            resnet.fc = nn.Identity()
            self.backbone = resnet
        elif backbone_name == "mobilenet_v2":
            mobilenet = models.mobilenet_v2(weights=None)
            feature_dim = mobilenet.classifier[1].in_features
            mobilenet.classifier = nn.Identity()
            self.backbone = mobilenet

        self.rnn_type = rnn_type
        if rnn_type == "LSTM":
            self.rnn = nn.LSTM(input_size=feature_dim, hidden_size=hidden_dim,
                               num_layers=num_rnn_layers, batch_first=True,
                               bidirectional=True, dropout=dropout if num_rnn_layers > 1 else 0)
        elif rnn_type == "GRU":
            self.rnn = nn.GRU(input_size=feature_dim, hidden_size=hidden_dim,
                              num_layers=num_rnn_layers, batch_first=True,
                              bidirectional=True, dropout=dropout if num_rnn_layers > 1 else 0)

        rnn_out_dim = hidden_dim * 2
        self.classifier = nn.Sequential(
            nn.Linear(rnn_out_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        if x.dim() == 5 and x.shape[-1] == 3:
            x = x.permute(0, 1, 4, 2, 3)

        batch_size, seq_len, C, H, W = x.shape
        x = x.reshape(batch_size * seq_len, C, H, W)
        cnn_features = self.backbone(x)

        rnn_input = cnn_features.reshape(batch_size, seq_len, -1)
        rnn_output, _ = self.rnn(rnn_input)
        temporal_feature = torch.mean(rnn_output, dim=1)

        logits = self.classifier(temporal_feature)
        return logits

# Cell 7: Tải Checkpoint Mô hình

In [99]:
checkpoint = torch.load(MODEL_PATH, map_location="cpu")
MODEL_CONFIG = checkpoint["config"]

model = SpatialTemporalFERModel(
    backbone_name=MODEL_CONFIG["backbone"],
    rnn_type=MODEL_CONFIG["rnn_type"],
    hidden_dim=MODEL_CONFIG["hidden_dim"],
    num_rnn_layers=MODEL_CONFIG["num_rnn_layers"],
    dropout=MODEL_CONFIG["dropout"],
    num_classes=MODEL_CONFIG["num_classes"]
)

model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(device)
model.eval()
print("✅ Đã nạp thành công mô hình FER!")

✅ Đã nạp thành công mô hình FER!


# Cell 8: Bộ xử lý Frame Buffer & Dự đoán Realtime/Batch

In [100]:
import time
import matplotlib.pyplot as plt

model_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Các biến trạng thái toàn cục
frame_buffer = []
timeline_history = []
last_process_time = 0
last_pred_emotion = "NEUTRAL"
last_pred_conf = 0.0
last_chart_fig = None

def process_single_frame_fast(input_frame):
    """
    Xử lý Frame Realtime với kỹ thuật Frame-Skipping & Throttle
    Giúp giảm lag tối đa và chống Spam 1 biểu cảm
    """
    global frame_buffer, timeline_history, last_process_time
    global last_pred_emotion, last_pred_conf, last_chart_fig

    if input_frame is None:
        return None, "CHỜ WEBCAM...", 0.0, pd.DataFrame(), None

    current_time = time.time()

    # ------------------------------------------------------------
    # 1. CẮT BỚT BẰNG TẦN SỐ (FPS Throttle - Tối đa 4 FPS để tránh giật)
    # Nếu chưa đủ 0.25 giây giữa 2 lần dự đoán -> Trả về kết quả cũ ngay lập tức
    # ------------------------------------------------------------
    if (current_time - last_process_time) < 0.25 and len(frame_buffer) > 0:
        # Vẽ nhanh Bounding Box lên frame hiện tại để giao diện mượt
        gray = cv2.cvtColor(input_frame, cv2.COLOR_BGR2GRAY)
        faces = face_cascade.detectMultiScale(gray, 1.2, 4, minSize=(40, 40))
        annotated = input_frame.copy()
        for (x, y, w, h) in faces:
            cv2.rectangle(annotated, (x, y), (x + w, y + h), (0, 255, 0), 2)
            cv2.putText(annotated, f"{last_pred_emotion} ({last_pred_conf:.1f}%)",
                        (x, max(20, y - 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

        df_hist = pd.DataFrame(timeline_history[-10:]) if timeline_history else pd.DataFrame()
        return annotated, last_pred_emotion, round(last_pred_conf, 2), df_hist, last_chart_fig

    last_process_time = current_time

    # ------------------------------------------------------------
    # 2. XỬ LÝ KHUÔN MẶT & CẬP NHẬT BUFFER
    # ------------------------------------------------------------
    gray = cv2.cvtColor(input_frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.1, 5, minSize=(40, 40))

    annotated_frame = input_frame.copy()
    cropped_face = input_frame.copy()

    if len(faces) > 0:
        x, y, w, h = max(faces, key=lambda r: r[2] * r[3])
        px, py = int(0.1 * w), int(0.1 * h)
        x1, y1 = max(0, x - px), max(0, y - py)
        x2, y2 = min(input_frame.shape[1], x + w + px), min(input_frame.shape[0], y + h + py)

        cropped_face = input_frame[y1:y2, x1:x2]
        cv2.rectangle(annotated_frame, (x, y), (x + w, y + h), (0, 255, 0), 2)

    # Chuẩn hóa màu sắc (Grading RGB) & Resize về kích thước 224x224
    rgb_face = cv2.cvtColor(cropped_face, cv2.COLOR_BGR2RGB)
    resized_face = cv2.resize(rgb_face, IMAGE_SIZE)

    frame_buffer.append(resized_face)
    if len(frame_buffer) > SEQUENCE_LENGTH:
        frame_buffer.pop(0)

    # Nếu chưa đủ 16 frames, nhân bản frame hiện tại để đủ độ dài Sequence
    temp_sequence = list(frame_buffer)
    while len(temp_sequence) < SEQUENCE_LENGTH:
        temp_sequence.append(resized_face)

    # ------------------------------------------------------------
    # 3. DỰ ĐOÁN MÔ HÌNH FER (FAST INFERENCE)
    # ------------------------------------------------------------
    tensors = [model_transform(f) for f in temp_sequence]
    video_tensor = torch.stack(tensors, dim=0).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(video_tensor)
        probs = torch.softmax(logits, dim=1)[0]

    top_idx = torch.argmax(probs).item()
    last_pred_emotion = CLASS_NAMES[top_idx].upper()
    last_pred_conf = float(probs[top_idx].item() * 100.0)

    # Đè chữ nhãn cảm xúc lên Bounding Box
    if len(faces) > 0:
        cv2.putText(annotated_frame, f"{last_pred_emotion} ({last_pred_conf:.1f}%)",
                    (x, max(20, y - 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

    # Ghi nhật ký Timeline
    sec_str = time.strftime("%H:%M:%S", time.localtime())
    timeline_history.append({
        "Thời gian": sec_str,
        "Cảm xúc (Emotion)": last_pred_emotion,
        "Độ tin cậy": f"{last_pred_conf:.1f}%"
    })
    df_timeline = pd.DataFrame(timeline_history[-10:])

    # ------------------------------------------------------------
    # 4. VẼ BIỂU ĐỒ PHÂN BỐ XÁC SUẤT
    # ------------------------------------------------------------
    fig, ax = plt.subplots(figsize=(5, 2.5))
    all_probs = [probs[i].item() * 100 for i in range(len(CLASS_NAMES))]
    bars = ax.bar(CLASS_NAMES, all_probs, color='#3182bd')
    bars[top_idx].set_color('#e6550d')
    ax.set_ylim(0, 100)
    ax.tick_params(axis="x", rotation=35, labelsize=7)
    plt.tight_layout()
    last_chart_fig = fig

    return annotated_frame, last_pred_emotion, round(last_pred_conf, 2), df_timeline, last_chart_fig

print("✅ Đã tối ưu hóa bộ xử lý Realtime Stream giảm giật lag!")

✅ Đã tối ưu hóa bộ xử lý Realtime Stream giảm giật lag!


# Cell 9: Giao diện Web Bố cục 3 Phần UX Chuẩn

In [101]:
import gradio as gr

with gr.Blocks(title="Facial Emotion Recognition Realtime Dashboard", theme=gr.themes.Soft()) as demo:

    gr.Markdown(
        """
        # 🎭 Hệ thống Nhận diện Biểu cảm Khuôn mặt (FER)
        ### Đồ án Công nghệ Phần mềm Nâng cao — Realtime Streaming Engine
        """
    )

    # 📌 PHẦN 1: BẢNG NẠP ĐẦU VÀO (ĐÃ THU NHỎ KÍCH THƯỚC)
    with gr.Accordion("⚙️ PHẦN 1: CẤU HÌNH & BẬT ĐẦU VÀO (WEBCAM / FILE VIDEO)", open=True):
        with gr.Row():
            # Cột trái: Chọn chế độ làm việc
            with gr.Column(scale=1):
                mode_radio = gr.Radio(
                    choices=["Webcam Trực Tiếp (Realtime)", "Tải Tập Tin Video (.mp4)"],
                    value="Webcam Trực Tiếp (Realtime)",
                    label="Chế độ nhận diện:"
                )
                process_file_btn = gr.Button("🚀 Phân tích Video File", variant="primary", visible=False)

            # Cột phải: Ô nhận nạp ảnh/video được thu gọn chiều cao (height=240)
            with gr.Column(scale=2):
                webcam_input = gr.Image(
                    sources=["webcam"],
                    streaming=True,
                    label="Khung Nhận Diện Webcam",
                    visible=True,
                    height=240  # Cố định chiều cao gọn gàng
                )
                file_input = gr.Video(
                    sources=["upload"],
                    label="Tải file Video (.mp4)",
                    visible=False,
                    height=240  # Cố định chiều cao gọn gàng
                )

    # 📌 PHẦN DƯỚI: KHÔNG GIAN BẢNG MÀN HÌNH THEO DÕI (BOTTOM MONITORING PANEL)
    with gr.Row():

        # 📌 PHẦN 2: MÀN HÌNH KẾT QUẢ (LEFT)
        with gr.Column(scale=1):
            gr.Markdown("### 🎬 PHẦN 2: Màn Hình Luồng Video / Live Stream")
            live_image_output = gr.Image(label="Live Stream Overlay Bounding Box", visible=True, height=320)
            file_video_output = gr.Video(label="Video đã gán nhãn", visible=False, height=320)

        # 📌 PHẦN 3: BẢNG THÔNG TIN XUYÊN SUỐT (RIGHT)
        with gr.Column(scale=1):
            gr.Markdown("### 📊 PHẦN 3: Bảng Theo Dõi & Analytics")

            with gr.Row():
                emotion_output = gr.Textbox(label="Cảm xúc hiện tại")
                confidence_output = gr.Number(label="Độ tin cậy (%)", precision=2)

            timeline_table = gr.Dataframe(
                headers=["Thời gian", "Cảm xúc (Emotion)", "Độ tin cậy"],
                label="⏱ Nhật ký phân tích",
                interactive=False
            )

            chart_output = gr.Plot(label="📈 Xác suất 8 lớp cảm xúc")

    # --- CHUYỂN ĐỔI GIAO DIỆN ---
    def switch_mode(choice):
        if choice == "Webcam Trực Tiếp (Realtime)":
            return (gr.update(visible=True), gr.update(visible=False),
                    gr.update(visible=False), gr.update(visible=True), gr.update(visible=False))
        else:
            return (gr.update(visible=False), gr.update(visible=True),
                    gr.update(visible=True), gr.update(visible=False), gr.update(visible=True))

    mode_radio.change(
        fn=switch_mode,
        inputs=mode_radio,
        outputs=[webcam_input, file_input, process_file_btn, live_image_output, file_video_output]
    )

    # --- LUỒNG WEBCAM STREAMING ---
    webcam_input.stream(
        fn=process_single_frame_fast,
        inputs=webcam_input,
        outputs=[live_image_output, emotion_output, confidence_output, timeline_table, chart_output],
        stream_every=0.25
    )

    # --- LUỒNG UPLOAD FILE ---
    process_file_btn.click(
        fn=process_uploaded_video,
        inputs=file_input,
        outputs=[file_video_output, emotion_output, confidence_output, timeline_table, chart_output]
    )

print("✅ Đã khởi tạo giao diện tối ưu kích thước!")

/tmp/ipykernel_2850/2498948361.py:6: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title="Facial Emotion Recognition Realtime Dashboard", theme=gr.themes.Soft()) as demo:


✅ Đã khởi tạo giao diện tối ưu kích thước!


# Cell 10: Khởi chạy Web Demo

In [102]:
print("🚀 Đang khởi chạy ứng dụng Web Demo...")
demo.launch(
    share=True,
    debug=True
)

🚀 Đang khởi chạy ứng dụng Web Demo...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://72ef7074c9dcd08b3b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.13/dist-packages/gradio/components/video.py:303: UserWarning: Video does not have browser-compatible container or codec. Converting to mp4.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/gradio/components/video.py:303: UserWarning: Video does not have browser-compatible container or codec. Converting to mp4.
  warnings.warn(


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://72ef7074c9dcd08b3b.gradio.live
